# CSE 4095 — Class 2: Classification
Use AI as a coding agent, but inspect and explain every major decision.

## 1. Load the Class 1 dataset
Upload `class1_student_habits.csv`, import Pandas, load it as `df`, and inspect the first rows.

In [1]:
import pandas as pd

# Only blank cells count as missing, so "None" stays a real
# prior_programming_experience answer instead of turning into NaN.
df = pd.read_csv("class1_student_habits.csv", keep_default_na=False, na_values=[""])

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Rows: 250, Columns: 10


,student_id,study_hours_per_week,sleep_hours_per_night,attendance_rate,practice_problems_per_week,prior_programming_experience,study_group,commute_minutes,quiz_average,exam_score
0,S001,1.3,7.7,71.8,8,Strong,No,0,67.0,64.6
1,S002,9.4,6.6,74.2,14,None,Yes,52,74.8,81.5
2,S003,11.2,6.1,70.3,15,Some,No,7,99.1,92.0
3,S004,10.8,7.6,77.1,16,Strong,No,12,86.4,100.0
4,S005,8.0,5.6,89.7,15,None,Yes,12,75.7,68.5


In [2]:
# Missing values by column (blank cells only)
df.isna().sum()

student_id                       0
study_hours_per_week             9
sleep_hours_per_night           12
attendance_rate                  0
practice_problems_per_week       0
prior_programming_experience     7
study_group                      0
commute_minutes                  0
quiz_average                     0
exam_score                       0
dtype: int64

## 2. Create the target
Create `high_exam_score`: Yes if `exam_score >= 80`, otherwise No. Then inspect class counts.

In [3]:
TARGET = "high_exam_score"
df[TARGET] = (df["exam_score"] >= 80).map({True: "Yes", False: "No"})

pd.DataFrame({
    "count": df[TARGET].value_counts(),
    "share": df[TARGET].value_counts(normalize=True).round(3),
})

,count,share
high_exam_score,,
No,151,0.604
Yes,99,0.396


## 3. Choose X and y
Use only legitimate features. Explain in Markdown why `student_id` and `exam_score` are excluded.

In [4]:
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
]
CATEGORICAL_FEATURES = [
    "prior_programming_experience",
    "study_group",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES]
y = df[TARGET]
print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (250, 6), y shape: (250,)


### Which columns would make the task artificially easy?

| Column | Verdict | Why |
|---|---|---|
| `exam_score` | **Leakage** | The target is computed directly from it. A model would just learn "score >= 80". |
| `quiz_average` | **Leakage** (for this task) | Measures the same skill as the exam (r = 0.77, far above any habit) and is not known when we only have habit data. |
| `student_id` | Excluded | An identifier with no real signal. A model could memorize individual students instead of learning patterns. |
| `commute_minutes` | Excluded | Not leakage, but almost no signal (r = 0.07) and not one of the intended features. |

The six features kept are all study habits or background known before the exam.

**Process leakage:** fitting imputers or scalers on the full dataset before splitting lets test rows influence training. The pipeline in Sections 5 and 6 prevents this.

## 4. Baseline
Calculate the majority-class baseline accuracy before training a model.

In [5]:
class_shares = y.value_counts(normalize=True)
majority_class = class_shares.idxmax()

print(f"Majority class: {majority_class}")
print(f"Always guessing '{majority_class}' is right {class_shares.max():.1%} of the time.")

Majority class: No
Always guessing 'No' is right 60.4% of the time.


**Why calculate the baseline first?**
- Accuracy means nothing without a reference point. 65% sounds fine until you learn that always guessing "No" gets 60%.
- The classes are imbalanced (99 Yes / 151 No). The baseline finds **0%** of the high scorers and still gets 60% accuracy, so accuracy alone can hide a useless model.
- A model that scores below the baseline points to a bug, such as flipped labels or a broken feature.

## 5. Train/test split and preprocessing
Ask your AI agent for a pipeline that handles numerical/categorical missing values and one-hot encoding.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

pd.DataFrame({
    "rows": [len(y_train), len(y_test)],
    "Yes": [(y_train == "Yes").sum(), (y_test == "Yes").sum()],
    "No": [(y_train == "No").sum(), (y_test == "No").sum()],
    "Yes_share": [(y_train == "Yes").mean().round(3), (y_test == "Yes").mean().round(3)],
}, index=["train", "test"])

,rows,Yes,No,Yes_share
train,200,79,121,0.395
test,50,20,30,0.400


**Why stratify?** The classes are imbalanced (40% Yes) and the test set is only 50 rows. Across 1,000 unstratified splits, the test set had anywhere from 11 to 30 "Yes" students, and 18% of splits were off by more than 4. `stratify=y` keeps both sets at about 40% Yes, so scores reflect the model and not a lucky split. `random_state` makes the split repeatable.

In [7]:
# Baseline scored the same way a real model will be:
# learn the majority class from TRAIN, score on TEST.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_accuracy = baseline_model.score(X_test, y_test)

print(f"Baseline always predicts: {baseline_model.predict(X_test)[0]}")
print(f"Baseline test accuracy:   {baseline_accuracy:.1%}")

Baseline always predicts: No
Baseline test accuracy:   60.0%


In [8]:
# What the preprocessing has to handle, looked at on TRAIN only
print("Missing values after the split:")
print(pd.DataFrame({"train": X_train.isna().sum(), "test": X_test.isna().sum()}))

print("\nFill values the imputers will learn (from X_train only):")
print(X_train[NUMERIC_FEATURES].median().round(2).to_string())
print(X_train[CATEGORICAL_FEATURES].mode().iloc[0].to_string())

print("\nCategories seen in training:")
for column in CATEGORICAL_FEATURES:
    train_categories = sorted(X_train[column].dropna().unique())
    test_only = set(X_test[column].dropna()) - set(train_categories)
    print(f"  {column}: {train_categories}, only in test: {sorted(test_only) or 'none'}")

Missing values after the split:
                              train  test
study_hours_per_week              6     3
sleep_hours_per_night            11     1
attendance_rate                   0     0
practice_problems_per_week        0     0
prior_programming_experience      5     2
study_group                       0     0

Fill values the imputers will learn (from X_train only):
study_hours_per_week           7.40
sleep_hours_per_night          7.10
attendance_rate               85.75
practice_problems_per_week    16.00
prior_programming_experience    Some
study_group                      Yes

Categories seen in training:
  prior_programming_experience: ['None', 'Some', 'Strong'], only in test: none
  study_group: ['No', 'Yes'], only in test: none


### Pipeline concepts

- **Transformer vs estimator:** transformers (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`) have `fit` (learn from data) and `transform` (apply it). Estimators (`LogisticRegression`) have `fit` and `predict`.
- **`Pipeline`:** chains steps in order. `fit` runs `fit_transform` on each transformer using the training data, then fits the model. `predict` only runs `transform` with what was already learned, then predicts. Test data never refits anything, which prevents process leakage.
- **`ColumnTransformer`:** sends numeric columns down one branch and categorical columns down another, then puts the results side by side (4 numeric + 3 experience + 2 study-group = 9 columns). Columns you don't list are dropped.
- **`SimpleImputer`:** fills missing values with a statistic learned during `fit`. The mean is pulled by outliers and the median is not. Text has no mean or median.
- **`StandardScaler`:** rescales each numeric column to mean 0 and standard deviation 1. Logistic regression penalizes every coefficient equally, so without scaling, columns with large units would be penalized unevenly. Scaled coefficients can also be compared with each other.
- **`OneHotEncoder`:** turns each category into its own 0/1 column. Mapping None/Some/Strong to 0/1/2 would assume the gaps between them are equal. `handle_unknown` controls what happens when the test set has a category train never saw.
- **Step names** (`"impute"`, `"classifier"`) are used for access (`model.named_steps[...]`), become column prefixes (`numeric__study_hours_per_week`), and are used for tuning (`classifier__C`).

The preprocessing is built in three steps below: a numeric branch, a categorical branch, and a `ColumnTransformer` that routes columns to them.

In [9]:
# Step 1: numeric branch (runs top to bottom on NUMERIC_FEATURES)
#   "impute": fill blanks with the TRAINING median. The median is used
#             instead of the mean so the practice_problems outlier (78)
#             can't pull the fill value up.
#   "scale":  rescale each column to mean 0, std 1 so logistic regression
#             treats columns with different units fairly.
numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# Step 2: categorical branch (runs top to bottom on CATEGORICAL_FEATURES)
#   "impute": fill blanks with the most common TRAINING answer. Text has
#             no mean or median, so "most_frequent" is the strategy that works.
#   "encode": one 0/1 column per category. handle_unknown="ignore" turns a
#             category never seen in training into all zeros instead of crashing.
categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

# Step 3: send each column list to its branch
#   Each entry is (name, pipeline, list_of_columns). Columns not listed
#   here are dropped, so the leaky columns can never reach the model.
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

## 6. Logistic regression
Fit the model using training data only.

In [10]:
# Step 4: full model = preprocessing, then the classifier
# Each step is a (name, object) tuple, and the model always goes last.
#   "preprocess": the ColumnTransformer from Step 3
#   "classifier": logistic regression. max_iter=1000 gives the solver
#                 enough steps to converge, random_state keeps runs repeatable.
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

# Step 5: fit on TRAIN only, then check the preprocessing output
model.fit(X_train, y_train)

transformed_train = model.named_steps["preprocess"].transform(X_train)
print(f"Shape after preprocessing: {transformed_train.shape}  (expected: (200, 9))")
print(f"Missing values left:       {pd.DataFrame(transformed_train).isna().sum().sum()}  (expected: 0)")
print("Columns the model sees:")
print(list(model.named_steps["preprocess"].get_feature_names_out()))

test_accuracy = model.score(X_test, y_test)
print(f"\nTest accuracy: {test_accuracy:.1%}  vs baseline {baseline_accuracy:.1%}")

Shape after preprocessing: (200, 9)  (expected: (200, 9))
Missing values left:       0  (expected: 0)
Columns the model sees:
['numeric__study_hours_per_week', 'numeric__sleep_hours_per_night', 'numeric__attendance_rate', 'numeric__practice_problems_per_week', 'categorical__prior_programming_experience_None', 'categorical__prior_programming_experience_Some', 'categorical__prior_programming_experience_Strong', 'categorical__study_group_No', 'categorical__study_group_Yes']

Test accuracy: 84.0%  vs baseline 60.0%


## 7. Evaluate
Calculate accuracy, confusion matrix, precision, and recall. Compare accuracy with the baseline.

In [11]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)

y_pred = model.predict(X_test)
baseline_pred = baseline_model.predict(X_test)


def summarize(y_true, y_predicted):
    """Accuracy plus precision and recall for the "Yes" class."""
    return {
        "accuracy": accuracy_score(y_true, y_predicted),
        "precision (Yes)": precision_score(y_true, y_predicted, pos_label="Yes", zero_division=0),
        "recall (Yes)": recall_score(y_true, y_predicted, pos_label="Yes", zero_division=0),
    }


pd.DataFrame({
    "baseline": summarize(y_test, baseline_pred),
    "logistic regression": summarize(y_test, y_pred),
}).round(3)

,baseline,logistic regression
accuracy,0.6,0.840
precision (Yes),0.0,0.875
recall (Yes),0.0,0.700


In [12]:
# Rows are the truth, columns are the model's guess.
labels = ["No", "Yes"]
pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"actual {label}" for label in labels],
    columns=[f"predicted {label}" for label in labels],
)

,predicted No,predicted Yes
actual No,28,2
actual Yes,6,14


In [13]:
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

          No      0.824     0.933     0.875        30
         Yes      0.875     0.700     0.778        20

    accuracy                          0.840        50
   macro avg      0.849     0.817     0.826        50
weighted avg      0.844     0.840     0.836        50



**Reading the results**

- **Accuracy:** 84.0% vs a 60.0% baseline. The model is right on 42 of 50 test students, 12 more than always guessing "No".
- **Confusion matrix:** 28 true No, 14 true Yes, **2 false positives** (predicted Yes, actually No) and **6 false negatives** (predicted No, actually Yes).
- **Precision (Yes) = 14 / (14 + 2) = 87.5%.** When the model says a student will score 80+, it is usually right.
- **Recall (Yes) = 14 / (14 + 6) = 70.0%.** The model finds 7 out of 10 high scorers and misses the other 3.
- The model leans toward "No", so its errors are mostly missed high scorers. That fits a 60/40 class split, where "No" is the safer guess.
- The baseline's precision and recall are both 0 because it never predicts "Yes". Its 60% accuracy comes entirely from the class imbalance.

## 8. Inspect individual predictions
Create a small table with actual class, predicted class, and predicted probability.

In [14]:
# predict_proba returns one column per class, in the order of model.classes_.
yes_column = list(model.classes_).index("Yes")

predictions = pd.DataFrame({
    "student_id": df.loc[X_test.index, "student_id"],
    "actual": y_test,
    "predicted": y_pred,
    "prob_yes": model.predict_proba(X_test)[:, yes_column].round(3),
})
predictions["correct"] = predictions["actual"] == predictions["predicted"]
predictions.head(10)

,student_id,actual,predicted,prob_yes,correct
113,S114,Yes,No,0.455,False
153,S154,No,No,0.152,True
216,S217,No,No,0.290,True
241,S242,No,No,0.404,True
31,S032,No,No,0.231,True
80,S081,Yes,No,0.490,False
140,S141,Yes,No,0.477,False
164,S165,No,No,0.312,True
77,S078,Yes,Yes,0.824,True
4,S005,No,No,0.161,True


In [15]:
# Every mistake, with the student's habits and real exam score for context.
# exam_score is joined here only to inspect errors. The model never saw it.
mistakes = (
    predictions[~predictions["correct"]]
    .join(X_test)
    .join(df["exam_score"])
    .sort_values("prob_yes")
)
print(f"{len(mistakes)} wrong out of {len(predictions)}")
mistakes

8 wrong out of 50


,student_id,actual,predicted,prob_yes,correct,study_hours_per_week,sleep_hours_per_night,attendance_rate,practice_problems_per_week,prior_programming_experience,study_group,exam_score
89,S090,Yes,No,0.298,False,NaN,6.2,72.7,15,Strong,No,82.0
99,S100,Yes,No,0.336,False,5.7,7.2,82.7,19,NaN,No,85.8
37,S038,Yes,No,0.342,False,5.5,7.1,78.3,22,Some,Yes,80.1
113,S114,Yes,No,0.455,False,10.3,6.0,89.5,9,Some,No,88.6
140,S141,Yes,No,0.477,False,9.4,7.0,82.8,13,Some,No,85.0
80,S081,Yes,No,0.490,False,5.6,7.6,87.3,36,None,No,87.9
169,S170,No,Yes,0.522,False,9.4,7.5,98.3,19,None,No,79.6
132,S133,No,Yes,0.523,False,9.7,5.7,92.9,13,Some,Yes,66.8


**What the individual predictions show**

- `prob_yes` is the model's probability that a student scores 80 or higher. The default rule predicts "Yes" when it is at least 0.5.
- **All 8 mistakes fall between 0.30 and 0.52.** The model is never confidently wrong. Its errors are borderline cases close to the 0.5 cutoff.
- Several misses are close to the 80-point line: S038 scored 80.1 and S170 scored 79.6. From habits alone these students look like everyone near the boundary.
- S090 is missing `study_hours_per_week` and S100 is missing `prior_programming_experience`. The imputer filled in typical values, which pulls these students toward average.
- S133 (predicted Yes, scored 66.8) is a genuine miss. Good habits (93% attendance, 9.7 study hours) did not lead to a high score, a reminder that habits only partly explain exam results.

## 9. Controlled leakage experiment
Temporarily include `exam_score` as a feature, compare results, and explain why this experiment is invalid as a real model.

In [16]:
def fit_and_score(numeric_columns):
    """Fit the same pipeline design on a different numeric column list.

    Uses the exact same train/test rows as the main model, so the only
    thing that changes between runs is which columns the model can see.
    """
    experiment_preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])
    experiment_model = Pipeline(steps=[
        ("preprocess", experiment_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    columns = numeric_columns + CATEGORICAL_FEATURES
    experiment_model.fit(df.loc[X_train.index, columns], y_train)
    experiment_pred = experiment_model.predict(df.loc[X_test.index, columns])
    return experiment_model, summarize(y_test, experiment_pred)


leaky_model, leaky_scores = fit_and_score(NUMERIC_FEATURES + ["exam_score"])

pd.DataFrame({
    "baseline": summarize(y_test, baseline_pred),
    "habits only (real model)": summarize(y_test, y_pred),
    "habits + exam_score (LEAKY)": leaky_scores,
}).round(3)

,baseline,habits only (real model),habits + exam_score (LEAKY)
accuracy,0.6,0.840,1.0
precision (Yes),0.0,0.875,1.0
recall (Yes),0.0,0.700,1.0


In [17]:
# Which column is the leaky model relying on? Numeric columns are scaled,
# so larger absolute coefficients mean more influence.
leaky_coefficients = pd.Series(
    leaky_model.named_steps["classifier"].coef_[0],
    index=leaky_model.named_steps["preprocess"].get_feature_names_out(),
)
leaky_coefficients.sort_values(key=abs, ascending=False).round(2)

numeric__exam_score                                 4.62
categorical__prior_programming_experience_None     -0.23
categorical__prior_programming_experience_Some      0.23
categorical__study_group_Yes                       -0.19
categorical__study_group_No                         0.19
numeric__sleep_hours_per_night                      0.12
numeric__practice_problems_per_week                 0.11
numeric__attendance_rate                           -0.06
numeric__study_hours_per_week                       0.01
categorical__prior_programming_experience_Strong    0.00
dtype: float64

**Why this experiment is invalid as a real model**

- **The result:** adding `exam_score` gives **100%** accuracy, precision and recall. The `exam_score` coefficient (4.62) is about 20 times larger than any habit, and `study_hours_per_week` drops to 0.01. The model stops using habits entirely.
- **It is circular.** `high_exam_score` is defined as `exam_score >= 80`, so the model only has to learn one cutoff. It learns nothing about students.
- **The feature doesn't exist at prediction time.** The point is to predict a result *before* the exam. If you already knew the exam score, you wouldn't need a model.
- **The warning sign:** a perfect or near-perfect score on a noisy, real-world problem usually means leakage. The honest result is the 84.0% habits-only model.
- The same reasoning applies to `quiz_average`. It's less extreme, but it measures the same skill as the exam, so it would inflate accuracy for the same reason.

## 10. Your experiment
Change one legitimate modeling decision. Write your prediction before running it, then compare results.

In [18]:
# TODO: student-designed experiment


## Conclusion
Write a short evidence-based conclusion and AI Reflection here.
